# Posterior summaries

Erik, 2026-11-18

A posterior is a distribution, and the draws are a sample from it. Every
summary in this notebook is computed the same way: do a calculation once per
draw, then summarize the results across draws. That covers the parameters
themselves, probabilities of statements about them, and quantities built from
them, such as the expected score for a given student or the variance the model
explains.

The notebook checks the summaries on a simulated fit with known parameters,
then applies them to the HSLS:09 draws saved by `05-sampling`. Run that
notebook first.

In [ ]:
import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

fw.scaffold()
import bayes  # functions/bayes.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")

clean = fw.data_read("inputs.final.hsls_clean", dtype={"student_id": str})
hsls = bayes.analysis_sample(clean)
regression = bayes.model("regression")

# The HSLS:09 draws and the settings they were sampled with, from 05-sampling
hsls_draws = fw.load_model("hsls_regression_draws")
settings = fw.load_model("hsls_regression_settings")
assert settings["n"] == len(hsls), "The saved draws come from a different sample. Rerun 05-sampling."

# One generator for the simulated students
rng = np.random.default_rng(707)
print(f"{len(hsls_draws):,} draws of {', '.join(c for c in hsls_draws.columns if c not in ('chain', 'draw'))}")

## Summaries of each parameter

The function below gives the usual summaries for each parameter:

- **Mean and median.** For a symmetric posterior they agree.
- **SD** of the draws, the Bayesian counterpart of a standard error.
- **Central intervals**: the 5th to 95th and 2.5th to 97.5th percentiles.
- **Highest-density interval (HDI)**: the narrowest interval holding 95% of the
  draws. For a symmetric, single-peaked posterior it matches the central
  interval; for a skewed one, such as the posterior of an SD, it shifts toward
  the mode. For a posterior with several peaks, the highest-density region can
  be split into pieces, and a single interval does not describe it.

The Monte Carlo standard error of each summary is in `06-diagnostics`, which
has the chain structure it needs.

In [ ]:
def hdi(values, level=0.95):
    # Slide a window holding the given share of the sorted draws, keep the
    # narrowest. A window from position i to i + width holds width + 1 draws.
    ordered = np.sort(np.asarray(values))
    width = int(np.ceil(level * len(ordered))) - 1
    lows, highs = ordered[: len(ordered) - width], ordered[width:]
    i = np.argmin(highs - lows)
    return lows[i], highs[i]


def summarize(draws, names=("alpha", "beta[1]", "beta[2]", "sigma")):
    rows = {}
    for name in names:
        values = draws[name]
        low90, high90 = bayes.interval(values, 0.9)
        low95, high95 = bayes.interval(values, 0.95)
        hdi_low, hdi_high = hdi(values)
        rows[name] = {
            "Mean": values.mean(),
            "Median": values.median(),
            "SD": values.std(),
            "5%": low90,
            "95%": high90,
            "2.5%": low95,
            "97.5%": high95,
            "HDI 95% low": hdi_low,
            "HDI 95% high": hdi_high,
        }
    return pd.DataFrame(rows).T

### Simulated data

1,000 simulated students with known parameters, fitted with the weak priors.

In [ ]:
truth = pd.Series({"alpha": 0.04, "beta[1]": 0.7, "beta[2]": 0.18, "sigma": 0.66}, name="Truth")
fake = bayes.simulate(1000, truth["alpha"], [truth["beta[1]"], truth["beta[2]"]], truth["sigma"], rng)
fake_draws = bayes.draws(bayes.sample(regression, bayes.stan_data(fake, bayes.WEAK)))
summarize(fake_draws).join(truth).round(3)

### HSLS:09

In [ ]:
summarize(hsls_draws).round(3)

## Probabilities of statements

A posterior gives probabilities for statements about parameters directly: the
share of draws for which the statement holds. Some statements about the
HSLS:09 slopes:

In [ ]:
beta1, beta2 = hsls_draws["beta[1]"], hsls_draws["beta[2]"]
statements = {
    "SES slope above 0": beta2 > 0,
    "SES slope above 0.15 SD per unit": beta2 > 0.15,
    "SES slope above 0.2 SD per unit": beta2 > 0.2,
    "Base-year math slope below 1": beta1 < 1,
    "Base-year math slope between 0.69 and 0.71": beta1.between(0.69, 0.71),
}
pd.DataFrame(
    {
        "Draws where it holds": {label: int(holds.sum()) for label, holds in statements.items()},
        "Posterior probability": {label: holds.mean() for label, holds in statements.items()},
    }
)

A probability of 1 here means all 4,000 draws satisfied the statement. The
posterior probability is then close to 1, not exactly 1, and 4,000 draws cannot
say how close. Statements near the edge of the posterior need many draws to
estimate well.

The base-year math slope below 1 is regression to the mean in relative
standing: a student who scored 1 SD (10 points) above 50 in the base year is
expected to score about 0.7 SD (7 points) above the intercept two years later,
given SES. Both
scores are T-scores, so this is about standing among classmates, not about
learning. Measurement error in the base-year score contributes to a slope below
1, as do real changes in standing; this regression cannot separate the two.

## Derived quantities

Any function of the parameters has a posterior too: compute it once per draw.
The cell computes the expected follow-up score, in SD units, for nine student
profiles: base-year scores of 40, 50, and 60 (−1, 0, and 1 SD) crossed with SES
of −1, 0, and 1. In the analysis sample, SES runs from about −1.9 to 2.9 with an
SD of 0.79.

In [ ]:
# Profiles are set as base-year T-scores and put in SD units with the saved
# constants. The expected follow-up score is in SD units.
profiles = pd.DataFrame([(math, ses) for math in [40, 50, 60] for ses in [-1, 0, 1]], columns=["w1_math", "w1_ses"])
profiles.insert(1, "w1_math_z", (profiles["w1_math"] - settings["math_center"]) / settings["math_scale"])

# One column per profile, one row per draw
expected = pd.DataFrame(
    {
        f"math {row.w1_math}, SES {row.w1_ses:+d}": hsls_draws["alpha"]
        + hsls_draws["beta[1]"] * row.w1_math_z
        + hsls_draws["beta[2]"] * row.w1_ses
        for row in profiles.itertuples()
    }
)
profiles["Posterior mean"] = expected.mean().to_numpy()
profiles[["5%", "95%"]] = np.array([bayes.interval(expected[c], 0.9) for c in expected.columns])
profiles.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for ses, group in profiles.groupby("w1_ses"):
    ax.errorbar(
        group["w1_math_z"] + ses * 0.06,
        group["Posterior mean"],
        yerr=[group["Posterior mean"] - group["5%"], group["95%"] - group["Posterior mean"]],
        fmt="o-",
        capsize=3,
        label=f"SES {ses:+d}",
    )
ax.set_xlabel("Base-year math (SD units; T-scores 40, 50, 60)")
ax.set_ylabel("Expected follow-up math (SD units)")
ax.set_title("Expected follow-up score by profile, posterior mean and 90% interval")
ax.legend()
plt.tight_layout()

The intervals are narrow: with 18,623 students, the expected score for a profile
is known within about 0.02 SD (0.2 points) either way. That is uncertainty about
the average student with the profile, not about any one student, whose score
varies around the expectation with an SD of about 0.66 (6.6 points).
`08-posterior-predictive` covers predictions for individual students.

## Contrasts and standardized effects

Three more derived quantities:

- **SES contrast**: the expected difference between two students with the same
  base-year score, one at SES +1 and one at SES −1, which is $2\beta_2$.
- **Standardized SES slope**: the change in the follow-up score, in follow-up
  SDs, for a one-SD change in SES, holding base-year math fixed. It is
  $\beta_2$ times the SD of SES over the SD of the follow-up score, using the
  sample SDs. The model's SD units come from the fixed T-score scale instead,
  so the two differ: the sample SD of SES is 0.79.
- **Bayesian $R^2$** (Gelman et al., 2019): for each draw, the variance of the
  expected scores across students over that variance plus $\sigma^2$. It gives
  a posterior for the share of variance the model explains.

In [ ]:
# The variance of the expected scores across students, for each draw: b' C b,
# where C is the covariance matrix of the predictors and b the draw's slopes
covariance = np.cov(hsls[bayes.PREDICTORS].to_numpy().T, bias=True)
slopes = hsls_draws[["beta[1]", "beta[2]"]].to_numpy()
mu_variance = np.einsum("dk,kl,dl->d", slopes, covariance, slopes)
derived = pd.DataFrame(
    {
        "SES contrast, +1 vs -1": 2 * hsls_draws["beta[2]"],
        "Standardized SES slope": hsls_draws["beta[2]"] * hsls["w1_ses"].std() / hsls["w2_math_z"].std(),
        "Bayesian R-squared": mu_variance / (mu_variance + hsls_draws["sigma"].to_numpy() ** 2),
    }
)
summarize(derived, names=derived.columns)[["Mean", "SD", "5%", "95%"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for name, ax in zip(derived.columns, axes):
    sns.histplot(derived[name], bins=50, stat="density", ax=ax)
    low, high = bayes.interval(derived[name], 0.9)
    ax.axvline(low, color="black", linestyle="--", linewidth=1)
    ax.axvline(high, color="black", linestyle="--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("")
axes[0].set_ylabel("Density")
fig.suptitle("Posteriors of derived quantities, with central 90% intervals (dashed)")
fig.tight_layout()

## Reporting

A report gives the estimate, its uncertainty, and the scale. The cell writes
the main result as a sentence from the draws, so the numbers in the text always
match the analysis.

In [ ]:
mean2 = hsls_draws["beta[2]"].mean()
low2, high2 = bayes.interval(hsls_draws["beta[2]"], 0.9)
contrast = derived["SES contrast, +1 vs -1"]
low_c, high_c = bayes.interval(contrast, 0.9)
print(
    f"Among students with the same base-year math score, each unit of SES goes with a follow-up "
    f"math score {mean2:.3f} SD higher (90% interval {low2:.3f} to {high2:.3f}), or "
    f"{bayes.to_points(mean2):.2f} T-score points. A student at SES +1 is expected to score "
    f"{contrast.mean():.2f} SD higher than a student at SES -1 (90% interval {low_c:.2f} to {high_c:.2f}), "
    f"or {bayes.to_points(contrast.mean()):.1f} points."
)

These are associations in the students with complete data, unweighted. They do
not say what would happen to a student's score if their family's SES changed,
and they leave out the students missing a math score or SES, who scored lower
on average (`01-hsls-descriptives`). Later notebooks in the series take up the
missing data and the design.